# Imports e df do drive


In [53]:
# Rode essa célula e reinicie a sessão
!pip install numpy==1.26.4
!pip install captum==0.8.0
!pip install shap==0.44.0
!pip install lime==0.2.0.1

In [54]:
import torch
import json
import numpy as np
import re
import pandas as pd
import torch
import shap
from IPython.display import display, HTML
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification,pipeline
from captum.attr import IntegratedGradients
from lime.lime_text import LimeTextExplainer
from sentence_transformers import util
import google.generativeai as genai
import json

# Carregar modelo e tokenizer
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = DistilBertTokenizer.from_pretrained(model_name)
model = DistilBertForSequenceClassification.from_pretrained(model_name)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [55]:
# Pegando as sentenças do DF

sentences_local = ["One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fact that it goes where other shows wouldn't dare. Forget pretty pictures painted for mainstream audiences, forget charm, forget romance...OZ doesn't mess around. The first episode I ever saw struck me as so nasty it was surreal, I couldn't say I was ready for it, but as I watched more, I developed a taste for Oz, and got accustomed to the high levels of graphic violence. Not just violence, but injustice (crooked guards who'll be sold out for a nickel, inmates who'll kill on order and get away with it, well mannered, middle class inmates being turned into prison bitches due to their lack of street skills or prison experience) Watching Oz, you may become comfortable with what is uncomfortable viewing....thats if you can get in touch with your darker side.",
"A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen- Michael Sheen not only has got all the polari but he has all the voices down pat too! You can truly see the seamless editing guided by the references to Williams' diary entries, not only is it well worth the watching but it is a terrificly written and performed piece. A masterful production about one of the great master's of comedy and his life. <br /><br />The realism really comes home with the little things: the fantasy of the guard which, rather than use the traditional 'dream' techniques remains solid then disappears. It plays on our knowledge and our senses, particularly with the scenes concerning Orton and Halliwell and the sets (particularly of their flat with Halliwell's murals decorating every surface) are terribly well done.",
"I thought this was a wonderful way to spend time on a too hot summer weekend, sitting in the air conditioned theater and watching a light-hearted comedy. The plot is simplistic, but the dialogue is witty and the characters are likable (even the well bread suspected serial killer). While some may be disappointed when they realize this is not Match Point 2: Risk Addiction, I thought it was proof that Woody Allen is still fully in control of the style many of us have grown to love.<br /><br />This was the most I'd laughed at one of Woody's comedies in years (dare I say a decade?). While I've never been impressed with Scarlet Johanson, in this she managed to tone down her sexy image and jumped right into a average, but spirited young woman.<br /><br />This may not be the crown jewel of his career, but it was wittier than Devil Wears Prada and more interesting than Superman a great comedy to go see with friends.",
"Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet & his parents are fighting all the time.<br /><br />This movie is slower than a soap opera... and suddenly, Jake decides to become Rambo and kill the zombie.<br /><br />OK, first of all when you're going to make a film you must Decide if its a thriller or a drama! As a drama the movie is watchable. Parents are divorcing & arguing like in real life. And then we have Jake with his closet which totally ruins all the film! I expected to see a BOOGEYMAN similar movie, and instead i watched a drama with some meaningless thriller spots.<br /><br />3 out of 10 just for the well playing parents & descent dialogs. As for the shots with Jake: just ignore them.",
"Petter Mattei's Love in the Time of Money is a visually stunning film to watch. Mr. Mattei offers us a vivid portrait about human relations. This is a movie that seems to be telling us what money, power and success do to people in the different situations we encounter. <br /><br />This being a variation on the Arthur Schnitzler's play about the same theme, the director transfers the action to the present time New York where all these different characters meet and connect. Each one is connected in one way, or another to the next person, but no one seems to know the previous point of contact. Stylishly, the film has a sophisticated luxurious look. We are taken to see how these people live and the world they live in their own habitat.<br /><br />The only thing one gets out of all these souls in the picture is the different stages of loneliness each one inhabits. A big city is not exactly the best place in which human relations find sincere fulfillment, as one discerns is the case with most of the people we encounter.<br /><br />The acting is good under Mr. Mattei's direction. Steve Buscemi, Rosario Dawson, Carol Kane, Michael Imperioli, Adrian Grenier, and the rest of the talented cast, make these characters come alive.<br /><br />We wish Mr. Mattei good luck and await anxiously for his next work.",
"Probably my all-time favorite movie, a story of selflessness, sacrifice and dedication to a noble cause, but it's not preachy or boring. It just never gets old, despite my having seen it some 15 or more times in the last 25 years. Paul Lukas' performance brings tears to my eyes, and Bette Davis, in one of her very few truly sympathetic roles, is a delight. The kids are, as grandma says, more like dressed-up midgets than children, but that only makes them more fun to watch. And the mother's slow awakening to what's happening in the world and under her own roof is believable and startling. If I had a dozen thumbs, they'd all be up for this movie.",
"I sure would like to see a resurrection of a up dated Seahunt series with the tech they have today it would bring back the kid excitement in me.I grew up on black and white TV and Seahunt with Gunsmoke were my hero's every week.You have my vote for a comeback of a new sea hunt.We need a change of pace in TV and this would work for a world of under water adventure.Oh by the way thank you for an outlet like this to view many viewpoints about TV and the many movies.So any ole way I believe I've got what I wanna say.Would be nice to read some more plus points about sea hunt.If my rhymes would be 10 lines would you let me submit,or leave me out to be in doubt and have me to quit,If this is so then I must go so lets do it.",
"This show was an amazing, fresh & innovative idea in the 70's when it first aired. The first 7 or 8 years were brilliant, but things dropped off after that. By 1990, the show was not really funny anymore, and it's continued its decline further to the complete waste of time it is today.<br /><br />It's truly disgraceful how far this show has fallen. The writing is painfully bad, the performances are almost as bad - if not for the mildly entertaining respite of the guest-hosts, this show probably wouldn't still be on the air. I find it so hard to believe that the same creator that hand-selected the original cast also chose the band of hacks that followed. How can one recognize such brilliance and then see fit to replace it with such mediocrity? I felt I must give 2 stars out of respect for the original cast that made this show such a huge success. As it is now, the show is just awful. I can't believe it's still on the air.",
"Encouraged by the positive comments about this film on here I was looking forward to watching this film. Bad mistake. I've seen 950+ films and this is truly one of the worst of them - it's awful in almost every way: editing, pacing, storyline, 'acting,' soundtrack (the film's only song - a lame country tune - is played no less than four times). The film looks cheap and nasty and is boring in the extreme. Rarely have I been so happy to see the end credits of a film. <br /><br />The only thing that prevents me giving this a 1-score is Harvey Keitel - while this is far from his best performance he at least seems to be making a bit of an effort. One for Keitel obsessives only.",
"If you like original gut wrenching laughter you will like this movie. If you are young or old then you will love this movie, hell even my mom liked it.<br /><br />Great Camp!!!"]

sentences = [re.sub(r'<br\s*/?>', ' ', s, flags=re.IGNORECASE) for s in sentences_local[0]] # Tamanho da lista de sentenças para processar
print(sentences)

NameError: name 'df' is not defined

# Shape Func

In [ ]:
def truncate_by_tokens(sentences, tokenizer, max_tokens=512):
    truncated = []
    for sentence in sentences:
        tokens = tokenizer.encode(sentence, truncation=True, max_length=max_tokens)
        # Decodifica de volta para texto (opcional, ou pode trabalhar com tokens)
        truncated_sentence = tokenizer.decode(tokens, skip_special_tokens=True)
        truncated.append(truncated_sentence)
    return truncated

truncated_sentences = truncate_by_tokens(sentences, tokenizer)

In [ ]:
# Criar pipeline de classificação
classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, top_k=None)

# Lime Func

In [ ]:
import numpy as np
# Definir labels
class_names = ["NEGATIVE", "POSITIVE"]
# Criar o explicador do LIME para texto
explainer_lime = LimeTextExplainer(class_names=class_names)
exp_objects = []

# Função de predição adaptada para o LIME (retorna numpy array com probs)
def predict_proba(texts):
    preds = classifier(texts)
    probs = []
    for p in preds:
        # Cria dicionário mapeando label → score
        label_to_score = {item['label']: item['score'] for item in p}
        # Garante ordem correta independente da saída do classifier
        prob_row = [label_to_score[label] for label in class_names]
        probs.append(prob_row)
    return np.array(probs)

def generate_lime_explanations_batch(sentences, batch_size=10):
    explanations = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        for j, sentence in enumerate(batch):
            exp = explainer_lime.explain_instance(sentence, predict_proba, num_features=20)
            explanations.append(exp.as_html())
            exp_objects.append(exp)
    return explanations, exp_objects

# IG Func

In [ ]:
# Colocar o modelo em modo avaliação
model.eval()

# Função de forward
def forward_func(embeds, attention_mask):
    outputs = model(inputs_embeds=embeds, attention_mask=attention_mask)
    probs = torch.softmax(outputs.logits, dim=-1)
    return probs[:, 1]  # classe POSITIVE

# Inicializar IG uma vez
ig = IntegratedGradients(forward_func)


# Move tensors to the GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 🔹 Função para processar sentenças com IG
def process_sentences(sentences, tokenizer, model, ig, max_length=128, n_steps=10):
    resultados = []
    delta = None
    device = next(model.parameters()).device  # Pega o device do modelo

    for text in sentences:
        # Tokenizar e mover para o mesmo device do modelo
        encodings = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
        input_ids = encodings["input_ids"].to(device)  # ✅ Move para GPU
        attention_mask = encodings["attention_mask"].to(device)  # ✅ Move para GPU

        # Obter embeddings
        embeddings = model.get_input_embeddings()(input_ids)

        # Baseline = embeddings zerados NO MESMO DEVICE
        baseline = torch.zeros_like(embeddings, device=device)  # ✅ Mesmo device

        # Calcular atribuições
        attributions, delta = ig.attribute(
            inputs=embeddings,
            baselines=baseline,
            additional_forward_args=(attention_mask,),
            return_convergence_delta=True,
            n_steps=n_steps
        )

        # Mapear atribuições para tokens (mover para CPU para numpy)
        tokens = tokenizer.convert_ids_to_tokens(input_ids[0].cpu())  # ✅ Move para CPU
        token_attribs = attributions[0].sum(dim=1).detach().cpu().numpy()  # ✅ Move para CPU

        # Guardar resultados (ignorando tokens especiais)
        for token, score in zip(tokens, token_attribs):
            if token not in ["[CLS]", "[SEP]", "[PAD]"]:
                resultados.append(f"{token:15} -> {score:.4f}")

        # Também pode guardar o delta se for útil
        delta = f"Convergence Delta: {delta.item():.4f}"

    return resultados, delta

# Bert Func


In [ ]:
def predict_sentiment(texts):
  # Tokenizar cada texto individualmente
  inputs = tokenizer(texts, return_tensors="pt", truncation=True, padding=True)

  # Mover os inputs para o mesmo dispositivo do modelo
  inputs = {k: v.to(model.device) for k, v in inputs.items()}

  with torch.no_grad():
      outputs = model(**inputs)

  prob = torch.nn.functional.softmax(outputs.logits, dim=-1)
  probabilities = prob.cpu().numpy().tolist()[0] # Convert to list for JSON serialization

  predicted_class = torch.argmax(prob, dim=1).item()

  sentiment = "POSITIVO" if predicted_class == 1 else "NEGATIVO"
  confidence = prob[0][predicted_class].item()

  return sentiment, confidence, probabilities

# Gerando as explicações

In [ ]:
# Gerar saida do Bert
lista_resultados_bert = []
for sentence in sentences:
  sentiment, confidence, probs = predict_sentiment(sentence)
  lista_resultados_bert.append((sentiment,confidence,probs))

In [ ]:
# Gerar explicações SHAP
# Definir explainer usando o pipeline (não direto no modelo cru)
explainer = shap.Explainer(classifier)
shap_values = explainer(truncated_sentences)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
PartitionExplainer explainer: 1762it [01:16, 20.98it/s]                          


In [ ]:
# Gera explicações LIME
lista_HTML_lime, lista_exp_lime = generate_lime_explanations_batch(sentences)
explanations_as_list = [exp.as_list() for exp in exp_objects]

ValueError: low >= high

In [ ]:
#  Gera explicações INTEGRATED GRADIENTS
# Move tensors to the GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
lista_resultados_ig = []
for i in sentences:
  lista_resultados, delta = process_sentences(i, tokenizer, model, ig)
  lista_resultados_ig.append((lista_resultados, delta))

In [ ]:
print("Bert Value")
for i in lista_resultados_bert:
  print(f"Sentence: {i[0]}")
  print(f"Confiança: {i[1]:.4f}")
  print(f"Probabilidades: [Negativo: {i[2][0]:.4f}, Positivo: {i[2][1]:.4f}]")

print("SHAP VALUES")
for i in shap_values:
  print(shap.plots.text(i))


print("\nLIME VALUES")
print("=" * 50)

for i, html in enumerate(lista_HTML_lime):
    print(f"\n→ Explicação {i+1}:")
    display(HTML(html))
    print("-" * 30)
print("Integrated Gradients")
for i in lista_resultados:
  print(i)

# Criando Json com saida dos Xai

In [ ]:
def generate_bert_json(sentiment, confidence, probs, index):
    try:
        # AGORA CRIAR O JSON COMPLETO
        Bert_data = {
            "explanations": {
                "sentiment": sentiment,
                "confidence": confidence,
                "probabilities": probs,
            },
            "sentence": sentences[int(index)]
        }


        with open('model_output'+ str(index) +'.json', 'w', encoding='utf-8') as f:
            json.dump(Bert_data, f, ensure_ascii=False, indent=2)

        print("✅ JSON completo salvo em 'model_output.json'")
    except Exception as e:
        print(f"❌ Ocorreu um erro ao gerar o JSON completo: {e}")
index = 0
for i in lista_resultados_bert:
  generate_bert_json(i[0], i[1], i[2], index)
  index = index + 1

In [ ]:
# Função para extrair dados SHAP de forma estruturada
def extract_shap_data(shap_values):
    shap_data = []
    negative_class_index = 0

    try:
        # Tenta acessar como se fosse um lote (batch)
        vals = shap_values.values[0]
        tokens = np.array(shap_values.data[0])
        base_vals = shap_values.base_values[0]
    except (IndexError, TypeError):
        # Se falhar, é porque o objeto já é a frase individual
        vals = shap_values.values
        tokens = np.array(shap_values.data)
        base_vals = shap_values.base_values

    # Extrai as contribuições para a classe negativa
    negative_shap_contributions = vals[:, negative_class_index]

    # Ordenação
    sorted_indices = np.argsort(np.abs(negative_shap_contributions))[::-1]
    sorted_tokens = tokens[sorted_indices]
    sorted_contributions = negative_shap_contributions[sorted_indices]

    for token, value in zip(sorted_tokens, sorted_contributions):
        if token.strip() != '':
            shap_data.append(f"  Token: '{token}' -> {value:.4f}")

    # Adiciona as probabilidades base (ajustado para evitar erro de índice)
    shap_data.append(f"Negative_prob = {base_vals[0]}")
    shap_data.append(f"Positive_prob = {base_vals[1]}")

    return shap_data

# Função para extrair dados LIME de forma estruturada
def extract_lime_data():
    out = []
    for sample in explanations_as_list:
        sample_list = []
        for token, value in sample:
            v = float(value)
            t = str(token)
            sample_list.append(f"{v:.6f} -> {t}")
        out.append(sample_list)
    return out

# Função para extrair dados IG de forma estruturada
def extract_ig_data():
    pat = re.compile(r"(#{1,2}[A-Za-z0-9]+|[A-Za-z0-9]+)\s*->\s*([-+]?\d*\.\d+|\d+)")

    out = []
    for s in lista_resultados:
        m = pat.search(str(s))
        if m:
            out.append({"token": m.group(1), "importance": float(m.group(2))})

    return out

def generate_xai_json(i):
    try:
        # AGORA CRIAR O JSON COMPLETO
        xai_complete_data = {
            "project_titled": "Increasing the reliability of AIs using XAI explanations for black-box model classification.",
            "model": "distilbert-base-uncased-finetuned-sst-2-english",
            "architectures": [
                "DistilBertForSequenceClassification"
            ],
            "xai_methods": ["SHAP", "LIME", "Integrated_Gradients"],
            "sentences": sentences,
            "explanations": {
                "bert": {
                    "sentiment": sentiment,
                    "confidence": confidence,
                    "probabilities": probs,
                },
                "shap": {
                    "shap_data_format": 'Token: "{token}" -> {value:.4f}',
                    "shap_data": [
                      "<shap>",
                      extract_shap_data(shap_values),
                      "</shap>"],
                },

                "lime": {
                    "lime_data_format": 'Token: "{token}" -> {value:.4f}',
                    "lime_data": [
                    "<lime>",
                    extract_lime_data(),
                    "</lime>"],
                },

                "integrated_gradients": {
                    "integrated_gradients_data_format": 'Token: "{token}", "importance": {value:.4f}}',
                    "integrated_gradients_data": [
                    "<ig>",
                    extract_ig_data(),
                    "</ig>"]
                }
            }
        }


        with open('XAI_output'+ str(i) +'.json', 'w', encoding='utf-8') as f:
            json.dump(xai_complete_data, f, ensure_ascii=False, indent=2)

        print("✅ JSON completo salvo em 'XAI_output.json'")
    except Exception as e:
        print(f"❌ Ocorreu um erro ao gerar o JSON completo: {e}")
for i in range(10):
  shap_values = auxiliar_lista_shap[i]
  explanations_as_list = auxiliar_lista_lime[i]
  lista_resultados = auxiliar_lista_ig[i]
  sentiment, confidence, probs = auxiliar_lista_bert[i]
  generate_xai_json(i)

In [ ]:

def generate_xai_json():
    try:
        # AGORA CRIAR O JSON COMPLETO
        xai_complete_data = {
            "metadata": {
                "project_titled": "Increasing the reliability of AIs using XAI explanations for black-box model classification.",
                "model": "distilbert-base-uncased-finetuned-sst-2-english",
                "activation": "gelu",
                "architectures": [
                  "DistilBertForSequenceClassification"
                ],
                "model_config": {
                    "attention_dropout": 0.1,
                    "dim": 768,
                    "dropout": 0.1,
                    "dtype": "float32",
                    "finetuning_task": "sst-2",
                    "hidden_dim": 3072,
                    "id2label": {
                      "0": "NEGATIVE",
                      "1": "POSITIVE"
                    },
                    "initializer_range": 0.02,
                    "label2id": {
                      "NEGATIVE": 0,
                      "POSITIVE": 1
                    },
                    "max_position_embeddings": 512,
                    "model_type": "distilbert",
                    "n_heads": 12,
                    "n_layers": 6,
                    "output_past": "true",
                    "pad_token_id": 0,
                    "qa_dropout": 0.1,
                    "seq_classif_dropout": 0.2,
                    "sinusoidal_pos_embds": "false",
                    "tie_weights_": "true",
                    "transformers_version": "4.57.1",
                    "vocab_size": 30522,
                },
                "tokenizer": "distilbert-base-uncased-finetuned-sst-2-english",
                "max_seq_length": 512,
                "dataset": "IMDB Dataset",
                "xai_methods": ["SHAP", "LIME", "Integrated_Gradients"],
            },
            "sentences": sentences,
            "explanations": {
                "bert": {
                    "sentiment": sentiment,
                    "confidence": confidence,
                    "probabilities": probs,
                },

                "shap": {
                    "shap_data_format": 'Token: "{token}" -> {value:.4f}',
                    "shap_data": [
                      "<shap>",
                      extract_shap_data(),
                      "</shap>"],
                },

                "lime": {
                    "lime_data_format": 'Token: "{token}" -> {value:.4f}',
                    "lime_data": [
                    "<lime>",
                    extract_lime_data(),
                    "</lime>"],
                },

                "integrated_gradients": {
                    "integrated_gradients_data_format": 'Token: "{token}", "importance": {value:.4f}}',
                    "integrated_gradients_data": [
                    "<ig>",
                    extract_ig_data(),
                    "</ig>"]
                }
            }
        }


        with open('xai_complete_analysis.json', 'w', encoding='utf-8') as f:
            json.dump(xai_complete_data, f, ensure_ascii=False, indent=2)

        print("✅ JSON completo salvo em 'xai_complete_analysis.json'")
    except Exception as e:
        print(f"❌ Ocorreu um erro ao gerar o JSON completo: {e}")

generate_xai_json()